# Extraction des Linéaments et Structures Géologiques par IA

## Introduction
Les linéaments sont des expressions rectilignes du terrain qui trahissent souvent des structures géologiques profondes comme des failles, des fractures ou des contacts lithologiques. Ce notebook utilise l'IA pour amplifier ces signaux structuraux souvent masqués par la végétation.

## Objectifs
*   **Ampliations des reliefs** : Utiliser Prithvi pour extraire les textures structurales.
*   **Détection géométrique** : Appliquer des filtres de gradient (Sobel/Canny) pour isoler les lignes directrices.
*   **Analyse de direction** : Identifier les familles de failles majeures dans notre zone d'étude au Congo.

## Méthodologie
1.  **Setup** : Installation de TerraTorch et d'OpenCV.
2.  **Acquisition** : Données Sentinel-2 (Bandes NIR et SWIR pour la structure).
3.  **Inférence** : Passage dans Prithvi pour obtenir des descripteurs de texture.
4.  **Filtrage** : Utilisation d'algorithmes de vision par ordinateur pour tracer les linéaments.

In [ ]:
# ====================================================
# ÉTAPE 1 : Setup
# ====================================================
!pip install geemap earthengine-api opencv-python scikit-learn rasterio terratorch torch matplotlib -q

import ee, geemap, torch, rasterio, cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from terratorch import BACKBONE_REGISTRY

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print('✅ Environnement prêt')

## Zone d'Étude (ROI)
Focus sur la zone régionale du Congo.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d'étude')
Map

## Acquisition des Données
Nous exportons les 6 bandes principales pour une analyse complète des textures multispectrales.

In [ ]:
# ====================================================
# ÉTAPE 3 : Acquisition satellite
# ====================================================
image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(roi).filterDate('2023-01-01', '2023-12-31')
         .median().clip(roi))

geemap.ee_export_image(image.select(['B2','B3','B4','B8','B11','B12']), 'struct.tif', scale=30, region=roi)

## Inférence Prithvi et Extraction de Linéaments
Nous utilisons le premier composant de la PCA sur les caractéristiques IA (qui porte souvent l'information structurale et topographique) puis appliquons un filtre de Canny pour isoler les lignes de fracture.

In [ ]:
# ====================================================
# ÉTAPE 4 : Inférence et Vision par Ordinateur
# ====================================================
model = BACKBONE_REGISTRY.build('prithvi_eo_v2_300', num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')
with rasterio.open('struct.tif') as src: img = src.read().astype(np.float32) / 10000.0

with torch.no_grad():
    out = model(torch.from_numpy(img).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out

feats_np = feats[0, 1:].numpy()
pca = PCA(n_components=1)
pc1 = pca.fit_transform(feats_np).reshape(int(np.sqrt(len(feats_np))), -1)

# Normalisation pour Canny
pc1_norm = cv2.normalize(pc1, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
edges = cv2.Canny(pc1_norm, 50, 150)

plt.figure(figsize=(12, 10))
plt.imshow(edges, cmap='gray_r')
plt.title('Carte des Linéaments Géologiques (IA + Canny)')
plt.axis('off')
plt.show()